Assignment2_FinNLP/
├── data/               # 存放下载好的数据集（如有）
├── models/             # 存放训练好的 LoRA 权重 (adapter_model.bin)
├── src/
│   ├── config.py       # 超参数配置
│   ├── data_loader.py  # 数据预处理脚本
│   ├── model_lib.py    # 封装 LoRA 注入逻辑
│   └── utils.py        # 绘图（混淆矩阵、Loss 曲线）
├── train_experiment.py # 运行实验的主脚本
└── evaluate_results.py # 生成对比表格和图表的脚本

In [4]:
import os
from peft import PeftModel

# 1. 使用你提供的绝对路径
absolute_path = "/Users/zimin/Develop/VScode_Project/Courses/Semester_2/EE7207_Deep_Learning/Assignments/A2/saved_models/checkpoint-729"

# 2. 物理检查：机理是确保 PEFT 所需的两个核心文件必须存在
config_exists = os.path.exists(os.path.join(absolute_path, "adapter_config.json"))
model_exists = os.path.exists(os.path.join(absolute_path, "adapter_model.bin")) or \
               os.path.exists(os.path.join(absolute_path, "adapter_model.safetensors"))

if config_exists and model_exists:
    print("✅ 物理文件校验通过，准备加载...")
    # 3. 加载模型
    model = PeftModel.from_pretrained(base_model, absolute_path)
    model.to(device)
    model.eval()
    print("🚀 LoRA 适配器挂载成功！")
else:
    print("❌ 校验失败！请检查文件夹内容：")
    if os.path.exists(absolute_path):
        print(f"文件夹内包含: {os.listdir(absolute_path)}")
    else:
        print("文件夹路径不存在，请检查路径拼写。")

✅ 物理文件校验通过，准备加载...
🚀 LoRA 适配器挂载成功！


In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
from config import Config

# 1. 确定设备
device = torch.device(Config.DEVICE)

# 2. 加载 Tokenizer (保持与训练时一致)
tokenizer = AutoTokenizer.from_pretrained(Config.BASE_MODEL_NAME)

# 3. 加载基础模型 (此时是未微调的状态)
base_model = AutoModelForSequenceClassification.from_pretrained(
    Config.BASE_MODEL_NAME, 
    num_labels=3
)

# 4. 挂载你训练好的 LoRA 适配器
# 注意：请将 'checkpoint-xxx' 替换为你 saved_models 目录下真实的文件夹名
model_path = "/Users/zimin/Develop/VScode_Project/Courses/Semester_2/EE7207_Deep_Learning/Assignments/A2/saved_models/checkpoint-729"
model = PeftModel.from_pretrained(base_model, model_path)
model.to(device)
model.eval() # 切换到推理模式，关闭 Dropout

print("模型已就绪！")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17124.17it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider t

✅ BERT + LoRA 模型已就绪！


In [10]:
def predict_sentiment(text):
    # 标签映射 (根据 Financial PhraseBank 的标准)
    id2label = {0: "Neutral", 1: "Positive", 2: "Negative"}
    
    # 1. 预处理：截断与填充至 128 长度
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        padding="max_length", 
        max_length=Config.MAX_LENGTH
    ).to(device)
    
    # 2. 推理：禁用梯度计算以节省显存
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        
    # 3. 后处理：使用 argmax 选出最高分的索引
    prediction = torch.argmax(logits, dim=-1).item()
    
    return id2label[prediction]

In [11]:
examples = [
    "Nvidia's revenue forecast beats analyst estimates, sending shares up 5%.", # Positive
    "The company announced a 10% reduction in its workforce due to declining sales.", # Negative
    "The central bank will meet next Tuesday to discuss the current monetary policy." # Neutral
]

# 你可以先用你的 LoRA 模型跑一下：
for text in examples:
    print(f"Result: {predict_sentiment(text)}")

Result: Positive
Result: Negative
Result: Neutral
